In [2]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q ultralytics lap

import sys, os, time, glob, pickle
import cv2, numpy as np, pandas as pd
from ultralytics import YOLO

def leer_frames(rutaVideo, tInicio, tFinal):
    cap = cv2.VideoCapture(rutaVideo)
    assert cap.isOpened()

    cap.set(cv2.CAP_PROP_POS_MSEC, tInicio * 1000)

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        t = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000
        if t > tFinal:
            break
        yield round(t * 50), t, frame

    cap.release()

def detectar_clip(ruta, tInicio, tFinal, mask, clases=(2, 3, 5, 7)):
    filas = []

    modelo = YOLO("yolo11n.pt")

    for idx, t, frame in leer_frames(ruta, tInicio, tFinal):
        res = modelo.track(frame, persist=True, tracker="bytetrack.yaml",
                   classes=list(clases), verbose=False)[0]
        cajas = res.boxes.xyxy.cpu().numpy()
        confianza = res.boxes.conf.cpu(). numpy()
        clase = res.boxes.cls.cpu().numpy().astype(int)

        if res.boxes.id is None:
            ids = np.full(len(cajas), -1)
        else:
            ids = res.boxes.id.cpu().numpy().astype(int)

        for (x1, y1, x2, y2), conf, cls, tids in zip(cajas, confianza, clase, ids):
            px = min(int((x1 + x2) / 2), mask.shape[1] - 1)
            py = min(int(y2), mask.shape[0] - 1)

            if mask[py, px] > 0:
                filas.append({
                    "frame": idx, "t": t,
                    "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                    "conf": conf, "cls": cls,
                    "id" : tids,
                })

    return pd.DataFrame(filas)

def lineas_medida(gt, ancho=1920):
    xs = np.arange(ancho)
    alturas = []
    for i in (4, 0):
        a, b, c = gt["measurementLines"][i]
        alturas.append(-(a * xs + c) / b)
    return alturas[0], alturas[1]

def tiempo_entre_lineas(tray, ln4, ln0):

    t4 = t0 = None

    primera = tray.iloc[0]
    if primera.y2 >= ln4[int((primera.x1 + primera.x2) / 2)]:
        return None

    for fila in tray.itertuples():
        x = int((fila.x1 + fila.x2) / 2)

        if t4 is None and fila.y2 >= ln4[x]:
            t4 = fila.t
        if t0 is None and fila.y2 >= ln0[x]:
            t0 = fila.t

    if t4 is None or t0 is None:
        return None

    return (t0, t4)

def calibrar_distancia(gt, t_max=1200):
    distancias = []
    for c in gt["cars"]:
        if c["valid"] and len(c["intersections"]) == 2 and c["intersections"][1]["videoTime"] < t_max:
            dt = c["intersections"][1]["videoTime"] - c["intersections"][0]["videoTime"]
            distancias.append(c["speed"] / 3.6 * dt)
    return np.median(distancias)

def emparejar(t4, gt_validos, tolerancia=0.2):
    mejor = min(gt_validos, key=lambda c: abs(c["intersections"][0]["videoTime"] - t4))
    if abs(mejor["intersections"][0]["videoTime"] - t4) < tolerancia:
        return mejor
    return None

def evaluar(res, gt_validos, tolerancia=0.2):
    parejas = [emparejar(t, gt_validos, tolerancia) for t in res["t4"]]
    res = res.copy()
    res["carId_gt"] = [p["carId"] if p else None for p in parejas]
    res["v_gt"]     = [p["speed"] if p else None for p in parejas]
    res["error"]    = res["velocidad"] - res["v_gt"]
    return res



d = "/content/drive/MyDrive/radar-trafico/data/raw/session2_center"
ruta = f"{d}/video.avi"
mask = cv2.imread(f"{d}/video_mask.png", cv2.IMREAD_GRAYSCALE)
with open(f"{d}/gt_data.pkl", "rb") as f:
    gt = pickle.load(f, encoding="latin1")

Y_LINEA_4, Y_LINEA_0 = lineas_medida(gt)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [3]:
SALIDA = "/content/drive/MyDrive/radar-trafico/data/processed/test"
os.makedirs(SALIDA, exist_ok=True)
TROZO, SOLAPE, T_INI, T_FIN = 60, 5, 1200, 3373

t_total = time.time()
for inicio in range(T_INI, T_FIN, TROZO):
    fichero = f"{SALIDA}/tracks_{inicio:04d}.csv"
    if os.path.exists(fichero):
        print("ya existe, se salta:", fichero)
        continue
    t0 = time.time()
    df_t = detectar_clip(ruta, inicio - SOLAPE, inicio + TROZO + SOLAPE, mask)
    df_t["trozo"] = inicio
    df_t.to_csv(fichero, index=False)
    print(f"trozo {inicio:4d} | {len(df_t):6d} filas | {time.time() - t0:5.0f} s")
print(f"Total: {(time.time() - t_total) / 60:.1f} min")

trozo 1200 |   2534 filas |   111 s
trozo 1260 |   3320 filas |    79 s
trozo 1320 |   3515 filas |    80 s
trozo 1380 |   2962 filas |    78 s
trozo 1440 |   2528 filas |    77 s
trozo 1500 |   1500 filas |    77 s
trozo 1560 |   1898 filas |    76 s
trozo 1620 |   2945 filas |    77 s
trozo 1680 |   3263 filas |    78 s
trozo 1740 |   3236 filas |    77 s
trozo 1800 |   2097 filas |    76 s
trozo 1860 |   2558 filas |    77 s
trozo 1920 |   2443 filas |    78 s
trozo 1980 |   3456 filas |    82 s
trozo 2040 |   3301 filas |    80 s
trozo 2100 |   2220 filas |    80 s
trozo 2160 |   3289 filas |    79 s
trozo 2220 |   2502 filas |    79 s
trozo 2280 |   2666 filas |    79 s
trozo 2340 |   2403 filas |    79 s
trozo 2400 |   1865 filas |    77 s
trozo 2460 |   2585 filas |    79 s
trozo 2520 |   3862 filas |    78 s
trozo 2580 |   4318 filas |    78 s
trozo 2640 |   3340 filas |    78 s
trozo 2700 |   2882 filas |    78 s
trozo 2760 |   3308 filas |    76 s
trozo 2820 |   1815 filas | 

In [4]:
import glob
ficheros = sorted(glob.glob(f"{SALIDA}/tracks_*.csv"))
assert len(ficheros) == 37, f"faltan trozos: solo hay {len(ficheros)}"

tracks = pd.concat([pd.read_csv(f) for f in ficheros], ignore_index=True)
tracks.to_csv(f"{SALIDA}/../tracks_test.csv", index=False)
print(len(tracks), "filas en total")

102246 filas en total


In [5]:
D = calibrar_distancia(gt, t_max=1200)          # D del bloque de AJUSTE, no de test
print(f"D = {D:.4f} m")

# 1. Velocidades del radar, trozo a trozo
filas = []
for (trozo, coche), tray in tracks[tracks["id"] >= 0].groupby(["trozo", "id"]):
    cruces = tiempo_entre_lineas(tray.sort_values("frame"), Y_LINEA_4, Y_LINEA_0)
    if cruces is None:
        continue
    t0, t4 = cruces
    if t0 > t4 and trozo <= t4 < trozo + TROZO:
        filas.append({"trozo": trozo, "id": coche, "t4": t4, "tiempo": t0 - t4,
                      "velocidad": D / (t0 - t4) * 3.6})
radar = pd.DataFrame(filas)

# 2. Emparejar con la gt del bloque de TEST
gt_test = [c for c in gt["cars"] if c["valid"] and len(c["intersections"]) == 2
           and c["intersections"][0]["videoTime"] >= 1200]
ev = evaluar(radar, gt_test)
ev.to_csv(f"{SALIDA}/../evaluacion_test.csv", index=False)

# 3. Cifras
emp = ev.dropna(subset=["carId_gt"])
print(f"Coches medidos por el radar:      {len(radar)}")
print(f"  emparejados con la gt:          {len(emp)}")
print(f"  sin pareja en la gt:            {len(radar) - len(emp)}")
print(f"  coches gt con 2+ IDs del radar: {emp['carId_gt'].duplicated().sum()}")
print(f"Cobertura: {emp['carId_gt'].nunique()} de {len(gt_test)} coches válidos de la gt "
      f"({emp['carId_gt'].nunique() / len(gt_test):.1%})")
print(f"Error medio absoluto: {emp['error'].abs().mean():.2f} km/h")
print(f"Error medio con signo: {emp['error'].mean():+.2f} km/h")
print(f"Coches con |error| < 3 km/h: {(emp['error'].abs() < 3).mean():.1%}")

D = 28.0000 m
Coches medidos por el radar:      1056
  emparejados con la gt:          957
  sin pareja en la gt:            99
  coches gt con 2+ IDs del radar: 25
Cobertura: 932 de 960 coches válidos de la gt (97.1%)
Error medio absoluto: 1.51 km/h
Error medio con signo: +0.79 km/h
Coches con |error| < 3 km/h: 95.2%
